In [ ]:
import openai, json
from dotenv import load_dotenv
load_dotenv()
client = openai.OpenAI()

In [ ]:
with open("test_emails.json", "r") as f:
    emails = json.load(f)

print(f"Total emails loaded: {len(emails)}")
print(json.dumps(emails[55], indent=2))

In [ ]:
# V0 Naive Classifier

def classify_email(prompt):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )
    return response.choices[0].message.content

In [ ]:
email = emails[0]
email['body']

In [ ]:

email = emails[1]
prompt = f"Classify this email: {email['body']}"
result = classify_email(prompt)
print(result)

Classify this email: I need to update my payment method for the subscription.
This email can be classified as a **Billing Inquiry** or **Account Management Request**.


In [ ]:
print("V0 OUTPUT (naive):")
print("-" * 40)
print(result)
print("-" * 40)
print(f"Expected : {email['correct_label']}")
print(f"Got      : {result[:50]}...")


# are you happy or priya is happy ??

In [13]:
# lets change the prompt
prompt = f"""You are an expert support email classifier 
for a SaaS product company.

Classify this email: {email['body']}

"""


result = classify_email(prompt)
print(result)

Classification: Billing Issue


In [15]:

result.split(':')[1]

' Billing Issue'

In [ ]:
for i in range(200):
    result = classify_email(emails)

In [17]:
# make it more feasable 

prompt = f"""You are an expert support email classifier 
for a SaaS product company.

Classify the email into EXACTLY ONE of these categories:
- Billing
- Technical  
- Feature Request
- Spam
- Other

Email: {email['body']}"""

result = classify_email(prompt)
print(result)

Billing


In [18]:
# lets enhance it more
prompt = f"""You are an expert support email classifier 
for a SaaS product company.

Classify the email into EXACTLY ONE of these categories:
- Billing
- Technical  
- Feature Request
- Spam
- Other

EMAIL TO CLASSIFY:
Subject: {email['subject']}
From: {email['sender']}
Body: {email['body']}"""


result = classify_email(prompt)
print(result)

Billing


In [ ]:
# still output is not looks goood ?


In [ ]:
prompt = f"""You are an expert support email classifier 
for a SaaS product company.

Classify the email into EXACTLY ONE of these categories:
- Billing
- Technical  
- Feature Request
- Spam
- Other

and also give Confidence Score (1-10)

Rules:
- Return ONLY the category name. Nothing else.
- No explanation, no punctuation, no extra words.
- If unsure, return "Other".



EMAIL TO CLASSIFY:
Subject: {email['subject']}
From: {email['sender']}
Body: {email['body']}

OUTPUT should be lile this only:
claasification | Confidence Score


"""




result = classify_email(prompt)
print(result)

Billing | 0.95


In [24]:
result

'Billing | 0.95'

In [ ]:
CLASSIFY_PROMPT = """You are an expert support email classifier 
for a SaaS product company.

Classify the email into EXACTLY ONE of these categories:
- Billing
- Technical  
- Feature Request
- Spam
- Other

Rules:
- Return ONLY the category name. Nothing else.
- No explanation, no punctuation, no extra words.
- If unsure, return "Other".

EMAIL TO CLASSIFY:
Subject: {subject}
From: {sender}
Body: {body}"""

In [ ]:
def classify_email(email):
    prompt = CLASSIFY_PROMPT.format(
        subject=email['subject'],
        sender=email['sender'],
        body=email['body']
    )
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    return response.choices[0].message.content.strip()


In [ ]:
# ALL 5 Emails, Track Score
correct = 0
failures = []

for email in emails[:10]:
    result = classify_email(email).strip()  # ← strip() added
    is_correct = result == email['correct_label']
    status = "✅" if is_correct else "❌"

    if is_correct:
        correct += 1        # ← ye missing tha!
    else:
        failures.append(email)  # ← ye bhi missing tha!

    print(f"{status} Expected: {email['correct_label']:16} Got: {result:16} | {email['subject'][:35]}")

print("-" * 55)
print(f"Score: {correct}/10 = {correct*10}% accuracy")


In [ ]:
# assignment hints
PROMPT_WITH_CONFIDENCE = """You are an expert support email classifier 
for a SaaS product company.

Classify the email into EXACTLY ONE of these categories:
- Billing
- Technical  
- Feature Request
- Spam
- Other

Also give a Confidence Score from 1-10:
- 10 = absolutely sure
- 5  = could go either way
- 1  = total guess

Rules:
- Return ONLY in this exact format:  Category | Score
- Example outputs:  Billing | 9
- Example outputs:  Technical | 6
- Example outputs:  Other | 3
- No explanation. Nothing else.

EMAIL TO CLASSIFY:
Subject: {subject}
From: {sender}
Body: {body}"""

In [ ]:
def build_classifier_prompt(subject, sender, body):
    return f"""
# ROLE
You are an expert support email classifier for a B2B SaaS company.
You have 10 years of experience triaging customer support emails.

# TASK  
Classify the incoming email into exactly one primary category,
one urgency level, and one sub-category where applicable.

# CATEGORIES
Primary (pick ONE):
- Billing
- Technical
- Feature Request
- Spam
- Other

Urgency (pick ONE):
- High    (customer cannot use the product)
- Medium  (issue exists but workaround available)
- Low     (question or minor inconvenience)

Billing Sub-category (only if Primary = Billing):
- Refund Request
- Failed Payment
- Pricing Question
- Invoice Issue
- Not Applicable

# RULES
- RETURN ONLY valid JSON. No explanation. No markdown.
- If the email is in any language other than English, still classify it.
- If you cannot determine the category, use "Other" and urgency "Low".

# OUTPUT FORMAT
Return ONLY this JSON structure, nothing else:
{{
  "category": "[Primary Category]",
  "urgency": "[Urgency Level]",
  "billing_sub": "[Sub-category or Not Applicable]"
}}

# EMAIL TO CLASSIFY
Subject: {subject}
From: {sender}
Body: {body}
"""


prompt = build_classifier_prompt(
    subject=email['subject'],
    sender=email['sender'],
    body=email['body']
)

def classify_email(prompt):        
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    return response.choices[0].message.content.strip()
result = classify_email(prompt)

# Any developer on your team can read 
# and modify this without breaking it — 
# that's maintainable prompt engineering.

In [ ]:
result

In [ ]:
long_thread = """
Hi Team,

My payment failed but I was charged twice. Please refund.

Thanks,
Mike

─────────────────────────────────────
From: Support Team
Thanks for reaching out! Can you share your account ID?

─────────────────────────────────────
From: Mike
Sure, account ID is ACC-4521. Also my team cannot login 
since this morning. Please fix urgently.

─────────────────────────────────────
From: Support Team  
We are looking into it. Meanwhile can you try clearing cache?

─────────────────────────────────────
From: Mike
Tried that. Still broken. Also wanted to ask — 
can you add a bulk export feature? 
Would be really helpful for our monthly reports.

─────────────────────────────────────
From: Billing Dept
We see the double charge. Refund has been initiated.

─────────────────────────────────────
From: Mike
Great. But login still broken. And what about the export feature?
Also our API rate limits keep hitting — payroll runs tonight.
Please help urgently.

─────────────────────────────────────
[Auto-generated footer]
This email and any attachments are confidential and intended 
solely for the use of the individual or entity to whom they 
are addressed. If you have received this email in error please 
notify the system manager. This message contains confidential 
information and is intended only for the individual named.
Please do not copy or forward this email to anyone else.
Thank you for your cooperation.
─────────────────────────────────────
""" * 3   # ← 3x repeat — make it very long

long_email = {
    "subject": "Re: Re: Re: Payment + Login + Feature",
    "sender": "mike@clientcorp.com",
    "body": long_thread,
    "correct_label": "Billing"
}

print(f"Email length: {len(long_thread)} characters")
print(f"Approx tokens: {len(long_thread) // 4}")

In [ ]:
def classify_email(email):
    prompt = CLASSIFY_PROMPT.format(
        subject=email['subject'],
        sender=email['sender'],
        body=email['body']
    )
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )
    return response.choices[0].message.content.strip()
result = classify_email(long_email)

In [ ]:
result

# you got wrong output

# START  → "payment failed, charged twice"  ← Billing signal
# MIDDLE → "login broken"                   ← Technical signal  
# MIDDLE → "bulk export feature"            ← Feature Request
# MIDDLE → "API rate limits"                ← Technical signal
# END    → "confidential footer..."         ← noise

In [ ]:
# crafting

email = {
    "subject": "Payment Error: Double Charge on Account",
    "sender": "mike@clientcorp.com",
    "body": "Wow great job charging me twice for this month's subscription. "
            "I was under the impression this was a tech company and not a bank. "
            "Please refund the amount wrongly charged."
}


PROMPT_STRUCTURED = """Return ONLY the category name.

Categories: Billing | Technical | Feature Request | Spam | Other

Subject: {subject}
Body: {body}"""


PROMPT_CRAFTED = """Return ONLY the category name.

Categories: Billing | Technical | Feature Request | Spam | Other

Rules:
- Sarcastic complaints ("wow great job")  still Billing
- Focus on ROOT CAUSE, not surface words
- "tech company not a bank"  ignore, still Billing

Subject: {subject}
Body: {body}
"""





In [ ]:
prompt_a = PROMPT_STRUCTURED.format(
    subject=email['subject'],
    body=email['body']
)

prompt_b = PROMPT_CRAFTED.format(
    subject=email['subject'],
    body=email['body']
)

In [ ]:
# updated llm fucntion
def classify_email(prompt):        
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    return response.choices[0].message.content.strip()

In [ ]:
classify_email(prompt_b)

In [ ]:
email_hard = {
    "subject": "Evaluating if we stay on your platform",
    "sender": "ceo@bigclient.com",
    "body": "Hi, we've been using your product for 2 years. "
            "Lately we're evaluating alternatives. "
            "Main blockers: (1) No Zapier integration "
            "(2) No bulk export feature "
            "(3) Pricing jumped 40% at last renewal. "
            "Can we discuss?"
}

prompt_a = PROMPT_STRUCTURED.format(
    subject=email_hard['subject'],
    body=email_hard['body']
)

In [ ]:
result_a = classify_email(prompt_a)
print("Prompt A Result:", result_a)

In [ ]:
prompt_b = PROMPT_CRAFTED.format(
    subject=email_hard['subject'],
    body=email_hard['body']
)


result_b = classify_email(prompt_b)
print("Prompt B Result:", result_b)

In [ ]:
# techniques
email_hard = {
    "subject": "Evaluating if we stay on your platform",
    "sender": "ceo@bigclient.com",
    "body": "Hi, we've been using your product for 2 years. "
            "Lately we're evaluating alternatives. "
            "Main blockers: (1) No Zapier integration "
            "(2) No bulk export feature "
            "(3) Pricing jumped 40% at last renewal. "
            "Can we discuss?"
}

PROMPT_ZERO_SHOT = """Return ONLY the category name.
Categories: Billing | Technical | Feature Request | Spam | Other

Subject: {subject}
Body: {body}"""

prompt_a = PROMPT_ZERO_SHOT.format(
    subject=email_hard['subject'],
    body=email_hard['body']
)
result_a = classify_email(prompt_a)
print("Zero-Shot Prompt Result:", result_a)

In [ ]:
PROMPT_FEW_SHOT = """Return ONLY the category name.
Categories: Billing | Technical | Feature Request | Spam | Other

EXAMPLE 1:
Body: "App crashes when I click save button"
Output: Technical

EXAMPLE 2:
Body: "Charged twice this month, please refund"
Output: Billing

EXAMPLE 3:
Body: "We're thinking of leaving — pricing too high, 
       also missing some features we need"
Output: Billing

EXAMPLE 4:
Body: "Your product is great but missing Zapier and bulk export. 
       Also our renewal price jumped 40%. Considering alternatives."
Output: Billing

RULE: If email mentions pricing complaint alongside feature requests,
      ALWAYS classify as Billing — pricing = churn risk = Billing team.

NOW CLASSIFY:
Subject: {subject}
Body: {body}"""

prompt_b = PROMPT_FEW_SHOT.format(
    subject=email_hard['subject'],
    body=email_hard['body']
)

result_b = classify_email(prompt_b)
print("Few-Shot Prompt Result:", result_b)

In [ ]:
def build_few_shot_prompt(email_subject, email_body):
    # These are REAL examples from your actual ticket history
    examples = [
        {
            "subject": "Charged $299 instead of $99",
            "body": "I upgraded to the Pro plan but you charged me $299, the Enterprise price.",
            "output": {"category": "Billing", "urgency": "High", "billing_sub": "Failed Payment"}
        },
        {
            "subject": "SFDC integration not syncing",
            "body": "Our Salesforce integration stopped syncing leads 3 days ago. Last sync: Nov 12.",
            "output": {"category": "Technical", "urgency": "High", "billing_sub": "Not Applicable"}
        },
        {
            "subject": "Can you add Slack notifications?",
            "body": "Would love to get a Slack ping when a new lead comes in. Is this planned?",
            "output": {"category": "Feature Request", "urgency": "Low", "billing_sub": "Not Applicable"}
        },
        {
            "subject": "WIN $500 AMAZON GIFT CARD!!!",
            "body": "Click here to claim your prize NOW limited time offer!!!!",
            "output": {"category": "Spam", "urgency": "Low", "billing_sub": "Not Applicable"}
        },
    ]
    
    # Build the examples block
    examples_block = ""
    for i, ex in enumerate(examples, 1):
        examples_block += f"""
                        EXAMPLE {i}:
                        Subject: {ex['subject']}
                        Body: {ex['body']}
                        Output: {ex['output']}
                        """
    
    return f"""
            CRITICAL: Return ONLY valid JSON. No explanation. No markdown.

            # ROLE
            You are a support email classifier for a B2B SaaS company with deep 
            knowledge of our product and customer patterns.

            # EXAMPLES OF CORRECT CLASSIFICATIONS
            Study these examples carefully — they reflect OUR specific business context:
            {examples_block}

            # NOW CLASSIFY THIS EMAIL
            Subject: {email_subject}
            Body: {email_body}

            REMINDER: Return ONLY the JSON. Exactly 3 fields: category, urgency, billing_sub.
            """

# Test with the edge case that was failing before:
result = classify_email(
    build_few_shot_prompt(
        "Our SFDC won't sync",
        "Salesforce stopped pulling contacts since Tuesday."
    )
)

In [ ]:
result

In [ ]:
email = {
    "subject": "SFDC integration not syncing",
    "sender": "cto@bigclient.com",
    "body": "Our Salesforce integration stopped syncing leads 3 days ago. "
            "Last sync: Nov 12. Our sales team is blind — "
            "no new leads coming in. We're losing deals. "
            "This needs to be fixed TODAY."
}

# Step 1: Classify karo

PROMPT_ZERO_SHOT = """Return ONLY the category name.
Categories: Billing | Technical | Feature Request | Spam | Other

Subject: {subject}
Body: {body}"""

prompt_a = PROMPT_ZERO_SHOT.format(
    subject=email_hard['subject'],
    body=email_hard['body']
)
classified = classify_email(prompt_a)
print(f"Step 1 Classified: {classified}")

In [ ]:
PRIORITY_SCORER_PROMPT = """You are a Customer Success Manager
at a B2B SaaS company with 5 years experience.

You understand:
- Sales team blocked = revenue impact = TOP priority
- SFDC = Salesforce = enterprise CRM = big client
- "Losing deals" = immediate churn risk

Given this classified ticket:
Category : {category}
Subject  : {subject}
Body     : {body}

Return ONLY this exact format:
Priority : [1-10]/10
SLA      : [X] hours  
Reason   : [one line]
Escalate : [Yes/No] — [which team]"""

prompt_b = PRIORITY_SCORER_PROMPT.format(
    category=classified,
    subject=email['subject'],
    body=email['body']
)

result = classify_email(prompt_b)
print("Step 2 Priority Result:", result)

In [ ]:
DRAFTER_PROMPT = """You are a Senior Support Agent.

Tone     : urgent, empathetic, action-oriented
Length   : 3-4 sentences only
Must have: what you are doing RIGHT NOW to fix it

Ticket:
Category : {category}
Subject  : {subject}
Body     : {body}

Write response — Subject line + body only."""

prompt_c = DRAFTER_PROMPT.format(
    category=classified,
    subject=email['subject'],
    body=email['body']
)

result = classify_email(prompt_c)
print("Step 3 Drafted Response:")
print(result)

In [ ]:
# Company          Chain
# ─────────────────────────────────────────────
# Freshdesk        Classify → Route → Auto-reply
# Zomato           Classify → Urgency → Chef alert
# Razorpay         Classify → Risk Score → Block/Allow
# Github Copilot   Read code → Understand → Suggest

In [ ]:
# all together
def call_llm(prompt):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    return response.choices[0].message.content.strip()



print("Step 1: Classifying...")

step1_prompt = f"""Return ONLY the category name.
Categories: Billing | Technical | Feature Request | Spam | Other

Subject : {email['subject']}
Body    : {email['body']}"""

category = call_llm(step1_prompt)
print(f"Category: {category}")



# ─────────────────────────────
# STEP 2 — PRIORITY SCORE
# ─────────────────────────────
print("\nStep 2: Scoring priority...")

step2_prompt = f"""You are a Customer Success Manager.
Score the priority of this support ticket from 1-10.

Category : {category}
Subject  : {email['subject']}
Body     : {email['body']}

Return ONLY this format:
Priority : [score]/10
SLA      : [X] hours
Escalate : Yes/No"""

priority = call_llm(step2_prompt)
print(priority)



# ─────────────────────────────
# STEP 3 — DRAFT RESPONSE
# ─────────────────────────────
print("\nStep 3: Drafting response...")

step3_prompt = f"""You are a Senior Support Agent.
Write a short response — 3 sentences max.
Mention what action you are taking RIGHT NOW.

Category : {category}
Priority : {priority}
Subject  : {email['subject']}
Body     : {email['body']}"""

draft = call_llm(step3_prompt)
print(draft)
